# FLAG 2027 - Notebook 35 RUN-ALL: a face-RECOGNITION embedding (ArcFace) next to CLIP

Our face side is a single encoder (CLIP ViT-L/14), while the voice side has six. CLIP describes how a face looks (age, sex, ethnicity, style); a face-recognition model (ArcFace, InsightFace buffalo_l, trained on WebFace600K) describes WHO it is, which is what the classic face-voice papers (FOP, VGGFace features) use. This notebook tests, with system E's loss (SupCon L_out):

| | Face input |
|---|---|
| F0 (copied from NB33) | CLIP only |
| F1 | CLIP + ArcFace |
| F2 | ArcFace only |

Step 1 extracts ArcFace for the challenge train + dev faces and all MAV rows (~43k images, ~30-40 min) after checking that the face files reproduce our stored CLIP features. Step 2 runs CV stage A and stage B (~25 min). Adopt only if stage A improves AND stage B passes the rule. If F1 wins, stage 2 re-extracts the VoxCeleb face frames (videos only, no audio) and NB20 gets system F. Frozen model, declared; training-side only, rules-safe.

**How to run:** T4 GPU, then Runtime > Run all. The first run installs InsightFace (~3-5 min). About 1-1.5 h in total. Resumable. Paste the **SUMMARY**.

In [ ]:
# Cell 1 [v1 4-Nov] - Drive, GPU (hard check), InsightFace (ArcFace face-recognition model, used FROZEN; declared in the system description)
import os, importlib.util
if importlib.util.find_spec('insightface') is None:
    os.system('pip -q install insightface onnxruntime-gpu')        # builds a small C extension, ~3-5 min the first time
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil, onnxruntime
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[v1 4-Nov] Device:', torch.cuda.get_device_name(0), '| onnxruntime providers:', onnxruntime.get_available_providers(), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 [v1 4-Nov] - FACE file lists in our row order, alignment check against the stored CLIP features, ArcFace extraction (resumable per split)
import os, re, json, time, zipfile, glob
import numpy as np, pandas as pd, cv2
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
PARTS = os.path.join(FEATDIR, 'arcface_parts'); os.makedirs(PARTS, exist_ok=True)
EXTRACT_NOTES = []
def enote(msg):
    print(msg); EXTRACT_NOTES.append(msg)
# ---- challenge train + dev faces ----
FULL = '/content/flag_faces'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png', '.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def jpg_token(tokens):
    t = [x for x in tokens if x.lower().endswith(('.jpg', '.jpeg', '.png'))]; assert len(t) == 1, tokens; return t[0]
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(l.split()[3], lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(l.split()[2], lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES:
    assert None not in FILES[s], 'missing face files in %s - tell Claude' % s
# ---- MAV v1/v3 faces (same rows as mavceleb_v1v3.npz) ----
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mavf'; mav_paths, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(fm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
# ---- MAV v2 + v1-complete 'extra' faces: the SAME random face per clip as the original extraction (same rng sequence as NB24/NB29) ----
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavxf/' + ver; flag = os.path.join(D, 'faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
assert [r[0] for r, k in zip(rows, keep) if k] == list(EXTRA['spk']), 'MAV extra row order differs - tell Claude'
# ---- alignment check: CLIP on 100 random face files per split must reproduce the stored CLIP features ----
from transformers import CLIPModel, CLIPProcessor
CLIPM = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); CLIPP = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
CLIP_SAVED = np.load(os.path.join(FEATDIR, 'clip.npz'))
stored = {'train': CLIP_SAVED['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}
for s in SPLIT_NAMES[1:]:
    if s in CLIP_SAVED.files:
        stored[s] = CLIP_SAVED[s]
rng_c = np.random.default_rng(1)
for s, saved in stored.items():
    assert len(saved) == len(FILES[s]), '%s: %d stored CLIP rows vs %d face files - tell Claude' % (s, len(saved), len(FILES[s]))
    idx = rng_c.choice(len(saved), 100, replace=False)
    x = CLIPP(images=[Image.open(FILES[s][i]).convert('RGB') for i in idx], return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    with torch.no_grad():
        new = CLIPM.visual_projection(CLIPM.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
    old = saved[idx]; cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    enote('alignment %-18s median CLIP cosine %.4f' % (s, np.median(cos))); assert np.median(cos) > 0.98, 'face files do not match the stored CLIP rows for %s - tell Claude' % s
del CLIPM; torch.cuda.empty_cache()
# ---- ArcFace (InsightFace buffalo_l: RetinaFace detector + ArcFace R50 recognition, 512-d), frozen ----
from insightface.app import FaceAnalysis
APP = FaceAnalysis(name='buffalo_l', root=os.path.join(CACHE, 'insightface'), allowed_modules=['detection', 'recognition'], providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
APP.prepare(ctx_id=0, det_size=(320, 320)); REC = APP.models['recognition']
def arc_one(p):
    img = cv2.imread(p)
    if img is None:
        img = cv2.cvtColor(np.array(Image.open(p).convert('RGB')), cv2.COLOR_RGB2BGR)
    h, w = img.shape[:2]; pad = int(0.25 * max(h, w))
    fs = APP.get(cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=0))   # padding helps the detector on tight face crops
    if fs:
        f = max(fs, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])); return f.normed_embedding.astype(np.float32), True
    e = REC.get_feat(cv2.resize(img, (112, 112))).reshape(-1).astype(np.float32)               # no face found: embed the whole (already face-centred) image
    return e / (np.linalg.norm(e) + 1e-9), False
ARC_OUT = {}
for s in ['train', 'mav', 'extra'] + SPLIT_NAMES[1:]:
    part = os.path.join(PARTS, s + '.npz')
    if os.path.exists(part):
        d = np.load(part); ARC_OUT[s], ARC_OUT['found_' + s] = d['e'], d['found']; continue
    t0 = time.time(); E = np.zeros((len(FILES[s]), 512), np.float32); found = np.zeros(len(FILES[s]), bool)
    for i, p in enumerate(tqdm(FILES[s], desc='arcface ' + s)):
        E[i], found[i] = arc_one(p)
    np.savez(part, e=E, found=found); ARC_OUT[s], ARC_OUT['found_' + s] = E, found
    print('  %s: %d faces in %.0f min' % (s, len(E), (time.time() - t0) / 60))
for s in ['train', 'mav', 'extra'] + SPLIT_NAMES[1:]:
    enote('arcface %-18s %6d images | face detected in %.1f%%' % (s, len(ARC_OUT[s]), 100 * ARC_OUT['found_' + s].mean()))
np.savez(os.path.join(FEATDIR, 'arcface.npz'), **ARC_OUT)
ytr_c = pd.read_csv(find_one('train_English_voices.csv'), header=None).iloc[:, 192].values.astype(int); Et = ARC_OUT['train'] / (np.linalg.norm(ARC_OUT['train'], axis=1, keepdims=True) + 1e-9)
ra = np.random.default_rng(3); a = ra.integers(len(Et), size=20000); b = ra.integers(len(Et), size=20000); same = ytr_c[a] == ytr_c[b]; cs = (Et[a] * Et[b]).sum(1)
enote('arcface sanity (challenge train): same-id cosine %.3f | different-id %.3f | gap %.3f' % (cs[same & (a != b)].mean(), cs[~same].mean(), cs[same & (a != b)].mean() - cs[~same].mean()))

In [ ]:
# Cell 2 - features, labels, CV folds (identical to NB24), harness with the new recipe options
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# Cell 5 - CV: does a face-RECOGNITION embedding (ArcFace) add to CLIP? All runs use system E's loss (SupCon L_out, w 0.5, tau 0.07).
# F0 = CLIP only (= NB33 L1_lout, copied, stage A and B) | F1 = CLIP + ArcFace concatenated | F2 = ArcFace only. Training-side only.
SUMMARY = list(EXTRACT_NOTES)
def lout_loss(ft, vt, lab, tau):
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def train_lout(F_raw, V_raw, all_tr, vsets, seed, w_al=0.5, tau=0.07):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
ARC = np.load(os.path.join(FEATDIR, 'arcface.npz'))
F_ARC = np.concatenate([l2(ARC['train']), l2(ARC['mav']), l2(ARC['extra'])]).astype(np.float32)
assert len(F_ARC) == len(F_RAW), 'arcface rows %d vs CLIP rows %d - tell Claude' % (len(F_ARC), len(F_RAW))
FACES = {'F1_clip+arcface': np.concatenate([F_RAW, F_ARC], 1), 'F2_arcface_only': F_ARC}
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV35 = os.path.join(RESDIR, 'nb35_cv.csv')
if not os.path.exists(CSV35):
    r = pd.read_csv(os.path.join(RESDIR, 'nb33_cv.csv')); r = r[r.combo == 'L1_lout'][['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender']].copy()
    r['combo'] = 'F0_clip_ref'; r.to_csv(CSV35, index=False); print('F0 copied from NB33 L1_lout (%d runs)' % len(r))
done35 = pd.read_csv(CSV35); t0 = time.time()
def run35(stage, names, seeds, fold_data):
    global done35
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done35.stage == stage) & (done35.combo == name) & (done35.seed == seed) & (done35.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]
                model, best_ep, XF, XV = train_lout(FACES[name], V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed)
                e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV35, mode='a', header=False, index=False); done35 = pd.concat([done35, row])
            print('%s %-16s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab35(stage):
    R = pd.read_csv(CSV35); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
run35('A', list(FACES), [1, 2, 3], FOLD_DATA)
run35('B', list(FACES), [4, 5, 6], FOLD_B)
TA, TB = tab35('A'), tab35('B'); refA, refB = TA.loc['F0_clip_ref'], TB.loc['F0_clip_ref']
note('NB35 STAGE A (usual folds, seeds 1-3; voice = system C stack + NAP; loss = SupCon L_out):'); note(TA.to_string())
note('NB35 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS (adopt only if stage A < 0 AND stage B: overall <= -0.4, or gender <= -0.5 with std <= +0.2):')
for c in FACES:
    dA, dB, dg, ds = TA.loc[c, 'overall'] - refA['overall'], TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    ok = dA < 0 and (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2))
    note('%s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, dA, dB, ds, dg, 'ADOPT' if ok else 'reject'))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)